In [1]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

# Load Function 6 data
X = np.load("function6/initial_inputs.npy")
Y = np.load("function6/initial_outputs.npy")
print("X shape:", X.shape)
print("Y shape:", Y.shape)


print(X)


print(Y)

X shape: (20, 5)
Y shape: (20,)
[[0.7281861  0.15469257 0.73255167 0.69399651 0.05640131]
 [0.24238435 0.84409997 0.5778091  0.67902128 0.50195289]
 [0.72952261 0.7481062  0.67977464 0.35655228 0.67105368]
 [0.77062024 0.11440374 0.04677993 0.64832428 0.27354905]
 [0.6188123  0.33180214 0.18728787 0.75623847 0.3288348 ]
 [0.78495809 0.91068235 0.7081201  0.95922543 0.0049115 ]
 [0.14511079 0.8966846  0.89632223 0.72627154 0.23627199]
 [0.94506907 0.28845905 0.97880576 0.96165559 0.59801594]
 [0.12572016 0.86272469 0.02854433 0.24660527 0.75120624]
 [0.75759436 0.35583141 0.0165229  0.4342072  0.11243304]
 [0.5367969  0.30878091 0.41187929 0.38822518 0.5225283 ]
 [0.95773967 0.23566857 0.09914585 0.15680593 0.07131737]
 [0.6293079  0.80348368 0.81140844 0.04561319 0.11062446]
 [0.02173531 0.42808424 0.83593944 0.48948866 0.51108173]
 [0.43934426 0.69892383 0.42682022 0.10947609 0.87788847]
 [0.25890557 0.79367771 0.6421139  0.19667346 0.59310318]
 [0.43216593 0.71561781 0.3418191  0.704

In [2]:
best_index = np.argmax(Y)

print("Best index:", best_index)
print("Best current x:", X[best_index])
print("Best current y:", Y[best_index])
print("Best current portal format:", "-".join(f"{v:.6f}" for v in X[best_index]))

kernel = C(1.0) * RBF(length_scale=0.2)

gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    random_state=42
)

gp.fit(X, Y)

rng = np.random.default_rng(42)
candidates = rng.uniform(0, 1, size=(10000, X.shape[1]))

mean, std = gp.predict(candidates, return_std=True)

kappa = 2.5
ucb = mean + kappa * std

best_ucb_index = np.argmax(ucb)
query = candidates[best_ucb_index]

print("Suggested query:", query)
print("Portal format:", "-".join(f"{v:.6f}" for v in query))
print("Predicted mean:", mean[best_ucb_index])
print("Predicted std:", std[best_ucb_index])
print("UCB score:", ucb[best_ucb_index])

Best index: 0
Best current x: [0.7281861  0.15469257 0.73255167 0.69399651 0.05640131]
Best current y: -0.7142649478202404
Best current portal format: 0.728186-0.154693-0.732552-0.693997-0.056401
Suggested query: [0.29648485 0.32208673 0.332446   0.7603676  0.04094436]
Portal format: 0.296485-0.322087-0.332446-0.760368-0.040944
Predicted mean: -0.6141774858086441
Predicted std: 0.3365994814823112
UCB score: 0.227321217897134


In [3]:
import numpy as np
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

# Load original Function 6 data
X = np.load("function6/initial_inputs.npy")
Y = np.load("function6/initial_outputs.npy")

# Add Week 1 query and output
week1_x = np.array([[0.296485, 0.322087, 0.332446, 0.760368, 0.040944]])
week1_y = np.array([-0.5070321901597052])

X = np.vstack([X, week1_x])
Y = np.append(Y, week1_y)

print("Updated X shape:", X.shape)
print("Updated Y shape:", Y.shape)

print("Best current x:", X[np.argmax(Y)])
print("Best current y:", np.max(Y))

# Fit GP surrogate model
kernel = C(1.0) * RBF(length_scale=0.2)

gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    random_state=42
)

gp.fit(X, Y)

# Generate candidate points
rng = np.random.default_rng(42)
candidates = rng.uniform(0, 1, size=(10000, X.shape[1]))

# Predict mean and uncertainty
mean, std = gp.predict(candidates, return_std=True)

# Expected Improvement acquisition
y_best = np.max(Y)
std_safe = std + 1e-12

improvement = mean - y_best
z = improvement / std_safe
ei = improvement * norm.cdf(z) + std_safe * norm.pdf(z)

best_ei_index = np.argmax(ei)
query = candidates[best_ei_index]

print("Acquisition used: Expected Improvement")
print("Suggested query:", query)
print("Portal format:", "-".join(f"{v:.6f}" for v in query))
print("Predicted mean:", mean[best_ei_index])
print("Predicted std:", std[best_ei_index])
print("EI score:", ei[best_ei_index])

Updated X shape: (21, 5)
Updated Y shape: (21,)
Best current x: [0.296485 0.322087 0.332446 0.760368 0.040944]
Best current y: -0.5070321901597052
Acquisition used: Expected Improvement
Suggested query: [0.35140444 0.33490169 0.5118298  0.85979972 0.14938995]
Portal format: 0.351404-0.334902-0.511830-0.859800-0.149390
Predicted mean: -0.4664911586057936
Predicted std: 0.1633936676175313
EI score: 0.08745140561350363


In [4]:
import numpy as np
from scipy.stats import norm

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C


# For Function 6, Week 2 improved over Week 1, but the best observed value is still negative.
# This suggests there may be a better region nearby, but the model should not become too exploitative.
# I use a balanced hybrid EI-UCB strategy with both global and local candidates.


# -----------------------------
# Load original Function 6 data
# -----------------------------
X = np.load("function6/initial_inputs.npy")
Y = np.load("function6/initial_outputs.npy")


# -----------------------------
# Add Week 1 and Week 2 results
# -----------------------------
week1_x = np.array([[0.296485, 0.322087, 0.332446, 0.760368, 0.040944]])
week1_y = np.array([-0.5070321901597052])

week2_x = np.array([[0.351404, 0.334902, 0.511830, 0.859800, 0.149390]])
week2_y = np.array([-0.39798887850658554])

X = np.vstack([X, week1_x, week2_x])
Y = np.append(Y, [week1_y[0], week2_y[0]])


print("Updated X shape:", X.shape)
print("Updated Y shape:", Y.shape)

best_index = np.argmax(Y)
best_x = X[best_index]
best_y = Y[best_index]

print("Best current x:", best_x)
print("Best current y:", best_y)


# -----------------------------
# Fit GP surrogate model
# -----------------------------
kernel = (
    C(1.0, (1e-3, 1e3))
    * Matern(length_scale=np.ones(X.shape[1]) * 0.2, length_scale_bounds=(1e-2, 1.0), nu=2.5)
    + WhiteKernel(noise_level=1e-8, noise_level_bounds=(1e-10, 1e-3))
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

gp.fit(X, Y)

print("Fitted kernel:", gp.kernel_)


# -----------------------------
# Generate candidate points
# -----------------------------
rng = np.random.default_rng(42)
dim = X.shape[1]

# Global candidates are important because Function 6 is still weak overall.
global_candidates = rng.uniform(0, 1, size=(25000, dim))

# Local candidates around the Week 2/current best point, since it improved over Week 1.
local_candidates = rng.normal(loc=best_x, scale=0.12, size=(15000, dim))
local_candidates = np.clip(local_candidates, 0, 1)

candidates = np.vstack([global_candidates, local_candidates])


# -----------------------------
# Predict mean and uncertainty
# -----------------------------
mean, std = gp.predict(candidates, return_std=True)

y_best = np.max(Y)
std_safe = std + 1e-12


# -----------------------------
# Expected Improvement
# -----------------------------
improvement = mean - y_best
z = improvement / std_safe
ei = improvement * norm.cdf(z) + std_safe * norm.pdf(z)


# -----------------------------
# Upper Confidence Bound
# -----------------------------
# Kappa is kept moderately high because this 5D function still needs exploration.
kappa = 2.0
ucb = mean + kappa * std


# -----------------------------
# Filtered hybrid EI-UCB
# -----------------------------
# Keep candidates that are either predicted to be competitive or have strong EI.
mean_filter = mean >= (y_best - 0.05)

if np.sum(mean_filter) == 0:
    mean_filter = mean >= np.percentile(mean, 90)

filtered_candidates = candidates[mean_filter]
filtered_mean = mean[mean_filter]
filtered_std = std[mean_filter]
filtered_ei = ei[mean_filter]
filtered_ucb = ucb[mean_filter]

ei_norm = (filtered_ei - np.min(filtered_ei)) / (np.max(filtered_ei) - np.min(filtered_ei) + 1e-12)
ucb_norm = (filtered_ucb - np.min(filtered_ucb)) / (np.max(filtered_ucb) - np.min(filtered_ucb) + 1e-12)

# Balanced weights because Function 6 improved, but is not solved.
hybrid_score = 0.55 * ei_norm + 0.45 * ucb_norm

best_hybrid_index = np.argmax(hybrid_score)
query = filtered_candidates[best_hybrid_index]


# -----------------------------
# Output results
# -----------------------------
print("Acquisition used: Filtered balanced Hybrid EI + UCB")
print("Number of candidates passing filter:", np.sum(mean_filter))
print("Suggested query:", query)

print("Portal format with hyphens:")
print("-".join(f"{v:.6f}" for v in query))

print("Portal format with x labels:")
print(",".join(f"x{i+1}:{v:.6f}" for i, v in enumerate(query)))

print("Predicted mean:", filtered_mean[best_hybrid_index])
print("Predicted std:", filtered_std[best_hybrid_index])
print("EI score:", filtered_ei[best_hybrid_index])
print("UCB score:", filtered_ucb[best_hybrid_index])
print("Hybrid score:", hybrid_score[best_hybrid_index])

Updated X shape: (22, 5)
Updated Y shape: (22,)
Best current x: [0.351404 0.334902 0.51183  0.8598   0.14939 ]
Best current y: -0.39798887850658554
Fitted kernel: 1.23**2 * Matern(length_scale=[0.54, 0.982, 1, 1, 0.999], nu=2.5) + WhiteKernel(noise_level=2.55e-10)
Acquisition used: Filtered balanced Hybrid EI + UCB
Number of candidates passing filter: 5664
Suggested query: [0.46995283 0.1334585  0.47574429 0.97355252 0.        ]
Portal format with hyphens:
0.469953-0.133458-0.475744-0.973553-0.000000
Portal format with x labels:
x1:0.469953,x2:0.133458,x3:0.475744,x4:0.973553,x5:0.000000
Predicted mean: -0.32166727939169015
Predicted std: 0.18349283654569998
EI score: 0.11760633211090149
UCB score: 0.0453183936997098
Hybrid score: 0.9301846091318725


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 1.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 1.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [5]:
week3_x = np.array([[
    0.469953,
    0.133458,
    0.475744,
    0.973553,
    0.000000
]])

week3_y = np.array([-0.862627252477251])

X = np.vstack([X, week1_x, week2_x, week3_x])
Y = np.append(Y, [
    week1_y[0],
    week2_y[0],
    week3_y[0]
])

best_index = np.argmax(Y)
best_x = X[best_index]
best_y = Y[best_index]

print("Shape:", X.shape, Y.shape)
print("Current best x:", best_x)
print("Current best y:", best_y)
print("Was Week 3 best?", best_index == len(Y) - 1)

Shape: (25, 5) (25,)
Current best x: [0.351404 0.334902 0.51183  0.8598   0.14939 ]
Current best y: -0.39798887850658554
Was Week 3 best? False


In [6]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel as C,
    Matern,
    WhiteKernel
)
import numpy as np

kernel = (
    C(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=np.full(5, 0.2),
        length_scale_bounds=(1e-2, 2.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-8,
        noise_level_bounds=(1e-10, 1e-3)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(X, Y)

print("Fitted kernel:", gp.kernel_)

/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 30 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 47 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 25 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/

Fitted kernel: 1.33**2 * Matern(length_scale=[0.751, 0.806, 1.29, 0.977, 0.842], nu=2.5) + WhiteKernel(noise_level=1e-10)


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


In [7]:
trust_radius = np.full(5, 0.12)

lower_bounds = np.maximum(0, best_x - trust_radius)
upper_bounds = np.minimum(1, best_x + trust_radius)

bounds = list(zip(lower_bounds, upper_bounds))

print("Lower bounds:", lower_bounds)
print("Upper bounds:", upper_bounds)

Lower bounds: [0.231404 0.214902 0.39183  0.7398   0.02939 ]
Upper bounds: [0.471404 0.454902 0.63183  0.9798   0.26939 ]


In [8]:
kappa = 0.75

def negative_ucb(point):
    point = np.asarray(point).reshape(1, -1)

    mean, std = gp.predict(point, return_std=True)
    ucb = mean[0] + kappa * std[0]

    # scipy minimises, so return the negative
    return -ucb

In [9]:
from scipy.optimize import minimize

rng = np.random.default_rng(42)

random_starts = rng.uniform(
    lower_bounds,
    upper_bounds,
    size=(120, 5)
)

starting_points = [best_x]
starting_points.extend(random_starts)

results = []

for start in starting_points:
    result = minimize(
        negative_ucb,
        x0=start,
        method="L-BFGS-B",
        bounds=bounds
    )

    if result.success:
        results.append(result)

if not results:
    raise RuntimeError("No successful optimisation runs")

best_result = min(results, key=lambda result: result.fun)
query = best_result.x

In [10]:
query_mean, query_std = gp.predict(
    query.reshape(1, -1),
    return_std=True
)

query_ucb = query_mean[0] + kappa * query_std[0]

distances = np.linalg.norm(X - query, axis=1)
nearest_index = np.argmin(distances)
nearest_distance = distances[nearest_index]

print("Method: Trust-region GP-UCB")
print("Suggested Week 4 query:", query)

print(
    "Portal format:",
    "-".join(f"{value:.6f}" for value in query)
)

print("Current best observed output:", best_y)
print("Predicted mean:", query_mean[0])
print("Predicted std:", query_std[0])
print("UCB:", query_ucb)
print("Distance to nearest observation:", nearest_distance)
print("Nearest observed point:", X[nearest_index])
print("Nearest observed output:", Y[nearest_index])

Method: Trust-region GP-UCB
Suggested Week 4 query: [0.44292868 0.40933251 0.63183    0.7398     0.12938114]
Portal format: 0.442929-0.409333-0.631830-0.739800-0.129381
Current best observed output: -0.39798887850658554
Predicted mean: -0.33876915699498666
Predicted std: 0.11232748459732657
UCB: -0.25452354354699175
Distance to nearest observation: 0.20764638668296645
Nearest observed point: [0.351404 0.334902 0.51183  0.8598   0.14939 ]
Nearest observed output: -0.39798887850658554
